# Daily Practice — 2026-09-23 — Pandas / Data Wrangling: Catching Split-Integrity Bugs Before They Reach Training

**Dataset:** [Census Income (Adult)](https://archive.ics.uci.edu/dataset/2/adult) — 48,842 real U.S. Census
records (age, workclass, education, occupation, hours worked, native country, and more) used to predict
whether a person earns `<=50K` or `>50K` per year. Loaded from the `jbrownlee/Datasets` public mirror of
the classic UCI dataset. This is genuinely messy real-world data: `"?"` sentinel values standing in for
missing data, 52 exact-duplicate rows, and inconsistent whitespace in string fields.

## Problem statement

You've moved from an SQA role onto a team about to train a model that predicts income bracket from this
Census data. Before anyone touches `model.fit()`, you've been asked to build the wrangling and validation
utilities that catch three specific bug classes that silently corrupt real training pipelines — the kind
of bugs that don't crash anything, they just quietly make your test-set metrics lie to you:

1. **Disguised missingness.** The raw CSV encodes missing values as the literal string `"?"`, not `NaN`.
   Left alone, `"?"` gets treated as a valid category by anything downstream (value counts, one-hot
   encoders, groupbys) and silently distorts every statistic that touches those columns.
2. **Row-level leakage from splitting before deduplication.** The raw data contains exact duplicate rows.
   If you `train_test_split` *before* removing them, a duplicate can land one copy in train and one copy
   in test — the model then gets evaluated on rows it already memorized, inflating test accuracy.
3. **Unseen categories at inference time.** Even with a clean split, a categorical column can have a value
   in the test set that never appeared in train (e.g. a rare `native_country`). A `OneHotEncoder` fit only
   on train will choke on it, and you want a check that surfaces this *before* it breaks a pipeline in
   production.

**What you should produce:** a small set of composable functions — a loader that fixes the sentinel-missing
problem, a missingness profiler, a deduplicator, a stratified splitter, a leakage detector, and an
unseen-category detector — then a final demonstration that runs the **wrong** workflow (split-then-dedup)
side by side with the **right** workflow (dedup-then-split) and shows, with real numbers, that the wrong
order leaks rows and the right order doesn't.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ADULT_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"

# The raw CSV has no header row; these are the standard UCI Adult/Census Income column names.
COLUMN_NAMES = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country",
    "income",
]

# String-typed columns, including the target — these are the ones that can carry "?" or stray whitespace.
CATEGORICAL_COLUMNS = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country", "income",
]

## 1. Load the data and fix disguised missingness

TODO: implement `load_data()`. It should:
- read `ADULT_URL` with `pd.read_csv`, passing `header=None` and `names=COLUMN_NAMES`
- replace every literal `"?"` value anywhere in the frame with `np.nan` (`DataFrame.replace` works
  frame-wide)
- strip leading/trailing whitespace from every column listed in `CATEGORICAL_COLUMNS`
- return the resulting `DataFrame`

Don't drop or impute anything here — this function's only job is "make missing values look like missing
values, and make string values comparable." Cleaning decisions come later.

In [ ]:
def load_data() -> pd.DataFrame:
    """Load the raw Census Income data with sentinel values converted to real NaNs."""
    # TODO
    raise NotImplementedError

raw_df = load_data()
raw_df.head()

## 2. Missingness profile

TODO: implement `profile_missingness(df)`. It should return a `DataFrame` indexed by column name, with
columns `n_missing` and `missing_rate`, containing **only the columns that have at least one missing
value**, sorted by `missing_rate` descending.

In [ ]:
def profile_missingness(df: pd.DataFrame) -> pd.DataFrame:
    """Return per-column missing counts and rates, for columns with at least one null."""
    # TODO
    raise NotImplementedError

profile_missingness(raw_df)

## 3. Deduplicate

TODO: implement `dedupe_and_report(df)`. It should drop exact duplicate rows (keeping the first occurrence),
reset the index, and return a tuple `(deduped_df, n_removed)` where `n_removed` is the integer number of
rows dropped.

In [ ]:
def dedupe_and_report(df: pd.DataFrame):
    """Drop exact duplicate rows. Returns (deduped_df, n_removed)."""
    # TODO
    raise NotImplementedError

deduped_df, n_removed = dedupe_and_report(raw_df)
print(f"removed {n_removed} duplicate rows")

## 4. Stratified train/test split

TODO: implement `stratified_split_no_leak(df, target_col, test_size, seed)`. Use
`sklearn.model_selection.train_test_split` with `stratify=df[target_col]` so the income-bracket balance is
preserved in both splits, and `random_state=seed` for reproducibility. Reset the index on both returned
frames and return `(train_df, test_df)`.

Note this function doesn't dedupe anything itself — it's a pure splitter. Whether the data it receives is
already deduplicated is the caller's responsibility, which is exactly the bug you're about to go hunting
for in section 7.

In [ ]:
def stratified_split_no_leak(df: pd.DataFrame, target_col: str, test_size: float, seed: int):
    """Stratified train/test split, reproducible via seed. Does not dedupe."""
    # TODO
    raise NotImplementedError

## 5. Leakage detector

TODO: implement `find_leaked_rows(train_df, test_df, subset=None)`. It should count how many rows in
`test_df` are exact duplicates (across the columns in `subset`, or all columns if `subset is None`) of some
row in `train_df`, and return that integer count. Hint: an inner merge of `test_df[cols]` against
`train_df[cols].drop_duplicates()` gives you exactly the leaked rows.

In [ ]:
def find_leaked_rows(train_df: pd.DataFrame, test_df: pd.DataFrame, subset: list = None) -> int:
    """Count rows in test_df that also appear (on `subset` columns) in train_df."""
    # TODO
    raise NotImplementedError

## 6. Unseen-category detector

TODO: implement `find_unseen_categories(train_df, test_df, cat_cols)`. For each column name in `cat_cols`,
compute the set of non-null values that appear in `test_df` but never appear in `train_df`. Return a `dict`
mapping column name -> set of unseen values, **including only columns that actually have at least one
unseen value** (skip columns with an empty unseen set).

In [ ]:
def find_unseen_categories(train_df: pd.DataFrame, test_df: pd.DataFrame, cat_cols: list) -> dict:
    """Map column -> set of category values present in test but absent from train."""
    # TODO
    raise NotImplementedError

## 7. Put it together: the buggy workflow vs. the correct one

TODO, using the functions above:

1. Run the **wrong** workflow: call `stratified_split_no_leak` directly on `raw_df` (i.e. split *before*
   deduplicating) with `target_col="income"`, `test_size=0.2`, `seed=42`. Call `find_leaked_rows` on the
   result and print the leak count — it should be **greater than 0**.
2. Run the **right** workflow: call `dedupe_and_report(raw_df)` to get `deduped_df`, then
   `stratified_split_no_leak(deduped_df, "income", 0.2, 42)`. Call `find_leaked_rows` on this result and
   print the leak count — it should be **exactly 0**.
3. On the correct split from step 2, call `find_unseen_categories` with `cat_cols=CATEGORICAL_COLUMNS` and
   print the result.
4. Print a one-paragraph-worth of summary lines: how many duplicate rows existed, how many rows leaked
   under the wrong workflow, and whether the correct workflow eliminated the leak.

In [ ]:
# TODO: put it all together here

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ADULT_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"

COLUMN_NAMES = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country",
    "income",
]

CATEGORICAL_COLUMNS = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country", "income",
]


def load_data() -> pd.DataFrame:
    df = pd.read_csv(ADULT_URL, header=None, names=COLUMN_NAMES)
    df = df.replace("?", np.nan)
    for c in CATEGORICAL_COLUMNS:
        df[c] = df[c].str.strip()
    return df


def profile_missingness(df: pd.DataFrame) -> pd.DataFrame:
    counts = df.isna().sum()
    counts = counts[counts > 0]
    rates = counts / len(df)
    return pd.DataFrame(
        {"n_missing": counts, "missing_rate": rates}
    ).sort_values("missing_rate", ascending=False)


def dedupe_and_report(df: pd.DataFrame):
    before = len(df)
    deduped = df.drop_duplicates().reset_index(drop=True)
    return deduped, before - len(deduped)


def stratified_split_no_leak(df: pd.DataFrame, target_col: str, test_size: float, seed: int):
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df[target_col]
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def find_leaked_rows(train_df: pd.DataFrame, test_df: pd.DataFrame, subset: list = None) -> int:
    cols = subset if subset is not None else list(train_df.columns)
    merged = test_df[cols].merge(train_df[cols].drop_duplicates(), how="inner")
    return len(merged)


def find_unseen_categories(train_df: pd.DataFrame, test_df: pd.DataFrame, cat_cols: list) -> dict:
    result = {}
    for c in cat_cols:
        train_vals = set(train_df[c].dropna().unique())
        test_vals = set(test_df[c].dropna().unique())
        unseen = test_vals - train_vals
        if unseen:
            result[c] = unseen
    return result


raw_df = load_data()
n_dupes = raw_df.duplicated().sum()

# Wrong workflow: split before dedup
wrong_train, wrong_test = stratified_split_no_leak(raw_df, "income", 0.2, 42)
leak_wrong = find_leaked_rows(wrong_train, wrong_test)

# Right workflow: dedup before split
deduped_df, n_removed = dedupe_and_report(raw_df)
right_train, right_test = stratified_split_no_leak(deduped_df, "income", 0.2, 42)
leak_right = find_leaked_rows(right_train, right_test)

unseen = find_unseen_categories(right_train, right_test, CATEGORICAL_COLUMNS)

print(f"duplicate rows in raw data: {n_dupes}")
print(f"leaked rows (split before dedup): {leak_wrong}")
print(f"leaked rows (dedup before split): {leak_right}")
print(f"unseen categories in correct split: {unseen}")
```

**Why this matters:** on this dataset the raw data has 52 exact duplicates, and splitting before
deduplicating leaks roughly a dozen rows into the "unseen" test set purely by chance — enough to nudge a
reported test accuracy without anyone noticing, especially on a model good enough to memorize its training
rows. Deduplicating first makes the leak count provably zero regardless of the random seed. The
unseen-category check comes back empty on this particular split, but the function is exactly the kind of
guardrail you'd wire into a CI pipeline: it costs nothing to run and it turns a `ValueError` at inference
time (or worse, a silent all-zeros one-hot column) into a caught issue at data-prep time.

</details>